In [1]:
##modules
#%matplotlib widget
%matplotlib inline
#
#%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf


from scipy.stats import permutation_test
sys.path.append(r"G:\apuntes_mne\code_MOUS\scripts_functions_created")
import re
# Ahora importa la función
from print5 import print5
# import pymer4 
# from pymer4.models import lmer, compare

import pickle

import re

In [2]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_SELF import get_paths_SELF

# Parámetros editables
disco = "g"
modality="visual"
layer_script = "event"
subj= "s01b"
type_epoch="self"

# Generar variables automáticamente
path_dict = get_paths_SELF(disco=disco, modality=modality,layer_script=layer_script,  subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")
    
    
with open(output_analysis / f"subjects_incomplete_{layer_script}.pkl", "rb") as f:
    subjects_remove = pickle.load(f)
    


✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\ICA_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_clean_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_matlab_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\evoked_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\channels_structure
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\raw_hsp
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\fwd
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\inverse
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_analysis\analysis_event\acw_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_analysis\analysis_eve

# Tables and paths
Now, as specially in the event_preprocessing we are going to work with lots of different comparisons, we are going to create, instead of variables only, we are going to use comparisons 

Example: comparison_1= [zinnen, woorden]
so variable 1= comparison_1[0]

then we will create a for loop, to print analysis for all conditions

### Paths
ACW_path
PLE_path

### Tables and metric_individuals
#### Block and event

f"acw_results_subjects_all_{layer_script}.pickle" 
if you want the acf and the whole table   f"autocorrelation_subjects_all_{filter_name}_{layer_script}.pickle"

f"autocorrelation_subjects_all_{layer_script}.pickle"

**variables** =  acw_50_elect_all_epoch_all //  'intercept_y_elect_all_epoch_all', 'slope_elect_all_epoch_all'


*FOOOF*
f"df_fooof_subject_all_{filter_name}_fixed_{layer_script}.pickle" 

**variables**
Band Powers
- 'delta'
- 'theta'
- 'alpha'
- 'beta'
- 'gamma'

Exponent 
- exponents            




#### dynamic

*ACW*


f"dynamic_autocorrelation_results_subjects_all_{layer_script}.pickle"

f"dynamic_autocorrelation_subjects_all_{layer_script}.pickle"

this is for FOOOF table_dynamic_fooof_results_all_filt_1-40_event

**variables** = acw_50_slope_elect_all_epoch_all   // acw_50_std_elect_all_epoch_all 


*FOOOF*

for all results of table f"table_dynamic_fooof_all_{suffix_str}.pickle"

for **only slopes and std**  
f"table_dynamic_fooof_results_all_{suffix_str}.pickle"
f"table_dynamic_fooof_results_all_{filtering}_{layer_script}.pickle"


**variables**
Band Powers
- 'delta_slope'
- 'theta_slope'
- 'alpha_slope'
- 'beta_slope'
- 'gamma_slope'

Exponents
- 'exponents_std'



# Table of items

**f"all_subjects_merged_events_tsv_stimuli.csv"**

This table contains the following columns
Index(['Subject', 'Condition', 'Epoch', 'Epoch_relative', 'Event_code',
       'order_trial', 'onset', 'Sample_events', 'Sample_tsv',
       'Sample_difference', 'Sample_aligned', 'Sentence_tsv', 'Text_stimuli',
       'number_item_original', 'number_item', 'par_item', 'match_ratio'],
      dtype='object')


We are going to be interested in number_item and par_item The rest of the columns are the indexes, or were used to build the table, like Sentence_tsv, so they will be removed 

Indexes will be adjusted later to check the number of epochs. 

In [3]:
filtering=True
if filtering==True:
    lfreq=1
    hfreq=40
    filter_name= f"filt_{lfreq}-{hfreq}"
    print(f"Filtrado aplicado: {lfreq}-{hfreq} Hz")
elif filtering==False:
    print("No se ha aplicado filtrado.")

Filtrado aplicado: 1-40 Hz


In [4]:
## Variables of script

# Modify this only if you want to use dynamic analysis, or static event analysis
if layer_script == "event":
    dynamic = False

elif layer_script == "block":
    dynamic = False  # It will ALWAYS be false in block


# -------------------------------
# Crop suffix
# -------------------------------
crop_epochs = None

if type_epoch == "emoc":
    crop_epochs = None


# Remove them also from table
# metric_individual NAMES, select which one you want depending on the ACW type,
# and whether it is dynamic or not
if dynamic == False:
    metrics = ["acw_50_elect_all_epoch_all", "acw_0_elect_all_epoch_all"]
    metric_individual = "acw_50_elect_all_epoch_all"

elif dynamic == True:
    metrics = ["acw_50_slope_elect_all_epoch_all", "acw_0_slope_elect_all_epoch_all"]
    metric_individual = "acw_50_slope_elect_all_epoch_all"


if "_elect_all_epoch_all" in metric_individual:
    print("yes")
    metric_individual_name = metric_individual.replace("_elect_all_epoch_all", "")
else:
    metric_individual_name = metric_individual


# -------------------------------
# Build ACW suffix
# Structure: type_epoch + filter_name + layer_script + crop
# -------------------------------
suffix_acw = []
suffix_acw.append(type_epoch)

if filtering:
    suffix_acw.append(filter_name)

suffix_acw.append(layer_script)

if crop_epochs is not None:
    suffix_acw.append(f"crop_{crop_epochs}")

suffix_acw_str = "_".join(suffix_acw)


# -------------------------------
# Build FOOOF suffix
# Structure: type_epoch + filter_name + aperiodic_mode + select_channels + layer_script + crop
# -------------------------------
aperiodic_mode = "fixed"

suffix_fooof = []
suffix_fooof.append(type_epoch)

if filtering:
    suffix_fooof.append(filter_name)

suffix_fooof.append(aperiodic_mode)
select_channels=None
if select_channels:
    suffix_fooof.append(select_channels)

suffix_fooof.append(layer_script)

if crop_epochs is not None:
    suffix_fooof.append(f"crop_{crop_epochs}")

suffix_fooof_str = "_".join(suffix_fooof)


## Variables of script
## path = analysis_path
path = ACW_path


# -------------------------------
# Read ACW table
# -------------------------------
if dynamic:
    name_table = f"dynamic_autocorrelation_results_subjects_all_{suffix_acw_str}.pickle"
else:
    name_table = f"acw_results_subjects_all_{suffix_acw_str}.pickle"

table_metric = pd.read_pickle(path / name_table)

print(f"name table metric is {name_table}")


# Extract subjects from dataframe
subjects = table_metric["Subject"].unique()

# Remove excluded subjects
subjects = [s for s in subjects if s not in subjects_remove]

# Filter table too
table_metric = table_metric.query("Subject in @subjects")


# -------------------------------
# Read FOOOF table
# -------------------------------
if dynamic:
    name_table_fooof = f"table_dynamic_fooof_results_all_{suffix_fooof_str}.pickle"
else:
    name_table_fooof = f"df_fooof_subject_all_{suffix_fooof_str}.pickle"

print(f"name table fooof is {name_table_fooof}")

table_fooof = pd.read_pickle(path / name_table_fooof)
table_fooof = table_fooof.query("Subject in @subjects")


# G:\PROYECTO_SELF\SELF_visual\output_analysis\analysis_event\acw_event
# df_fooof_subject_all_emoc_filt_1-40_fixed_event

# if layer_script == "event":
#     path_csv_table = datadir / "all_subjects_merged_events_tsv_stimuli.csv"
#     all_subjects_merged_events_tsv_stimuli = pd.read_csv(path_csv_table)
#     print("Table all_subjects_merged_events_tsv_stimuli read")

yes
name table metric is acw_results_subjects_all_self_filt_1-40_event.pickle
name table fooof is df_fooof_subject_all_self_filt_1-40_fixed_event.pickle


# Mix tables of events and ACW

# WE WONT´APPLY THIS FOR THE MOMENT
### Index of epochs  

Here i can merge the tables of  all_subjects_merged_events_tsv_stimuli  and the ACW, the problem is that they dont share **dimensionality**, because they ACW includes registrations for electrode. 

So we have to consider this the table. 

We are going to mix the tables using Subject, Condition, Epoch and Epoch_relative, to add the number of the epoch relative to EACH condition

Index of Epochs is going to be applied **ONLY IN EVENT LAYER (also in dynamic)**

In [5]:
table_metric.columns

Index(['Subject', 'event_id', 'Condition_self', 'Condition_emotion',
       'Condition_gaze', 'Epoch', 'Elect', 'acw_50_elect_all_epoch_all',
       'acw_0_elect_all_epoch_all'],
      dtype='object')

In [6]:
# first we add the Epoch relative to the table

def renumber_epochs(df):
    df = df.copy()
    group_cols = ["Subject","Condition_self", "Condition_emotion", "Condition_gaze"]

    df["Epoch_relative"] = (
        df.groupby(group_cols)["Epoch"]
        .transform(lambda x: x.map({v: i for i, v in enumerate(sorted(pd.unique(x)))}))
    )
    return df



In [7]:



table_metric_r = renumber_epochs(table_metric)

In [8]:
# if layer_script=="event":
#     print(f"all_subjects_merged_events_tsv_stimuli.columns are {all_subjects_merged_events_tsv_stimuli.columns}")

# print(f"table_metric_r.columns are {table_metric_r.columns}")

In [9]:
# if layer_script=="event":
#     # now we merge both tables, 

#     keys = ["Subject", "Condition_self", "Condition_emotion", "Condition_gaze", "Epoch", "Epoch_relative"]

#     table_metric_with_stimuli = table_metric_r.merge(
#         all_subjects_merged_events_tsv_stimuli,
#         on=keys,
#         how="left",        # conserva todas las filas de table_metric_r (por electrodo)
#         validate="m:1"     # muchas (electrodos) -> una (evento) por clave
#     )

#     table_metric_with_stimuli
    
    
#         # This checks the number of merges

#     col_check = "Sample_events"   # columna que venga solo del df stimuli

#     n_total = len(table_metric_with_stimuli)
#     n_match = table_metric_with_stimuli[col_check].notna().sum()
#     n_no_match = table_metric_with_stimuli[col_check].isna().sum()

#     print(f"Total filas: {n_total}")
#     print(f"✔️ Con merge: {n_match}")
#     print(f"❌ Sin merge: {n_no_match}")
    

# #now we rename them both, to maintain nomenclature
# if layer_script=="event":    # we use the table with the stimuli
#     table_metric_to_merge=table_metric_with_stimuli.copy()

# if layer_script=="block":   # we use only metric table
#     table_metric_to_merge=table_metric_r.copy()

    
    


# Mix tables of ACW and FOOOF

### Index of epochs  ACW and FOOOF
 
So I have a problem here, as FOOOF for dynamic events is using relative indexed for epochs, BUT ACW is using absolute indexes

For the LMM models, I have to know EXPLICITELY wich epoch (and item ) I am using, so check what are the indexes for

ACW
- Block
- Event
- Dynamic

FOOOF
- Block
- Event
- Dynamic



To solve this, Im going to use Epoch_relative, using **renumber_epochs** for table_fooof, although it was previously used also with table_metric_r


In [10]:

table_fooof_r = renumber_epochs(table_fooof)

# n_diff = (table_fooof_r["Epoch"] != table_fooof_r["Epoch_relative"]).sum()

# # So this print is just to check, and should be =0
# print("Rows where Epoch != Epoch_relative:", n_diff)

In [11]:
## this code checks differences in unique values of columns between both dataframes, table_metric_to_merge, and table_fooof_r


cols = ["Subject", "Condition_self", "Condition_emotion", "Condition_gaze","Epoch", "Epoch_relative", "Elect"]


for col in cols:
    set_df = set(table_metric_r[col].unique())
    set_FOOOF = set(table_fooof_r[col].unique())

    print(f"\n--- {col} ---")
    print("Solo en table_metric:", set_df - set_FOOOF)
    print("Solo en table_fooof:", set_FOOOF - set_df)


--- Subject ---
Solo en table_metric: set()
Solo en table_fooof: set()

--- Condition_self ---
Solo en table_metric: set()
Solo en table_fooof: set()

--- Condition_emotion ---
Solo en table_metric: set()
Solo en table_fooof: set()

--- Condition_gaze ---
Solo en table_metric: set()
Solo en table_fooof: set()

--- Epoch ---
Solo en table_metric: set()
Solo en table_fooof: set()

--- Epoch_relative ---
Solo en table_metric: set()
Solo en table_fooof: set()

--- Elect ---
Solo en table_metric: set()
Solo en table_fooof: set()


In [12]:

cols = ["Subject", "event_id", "Condition_self", "Condition_emotion", "Condition_gaze", "Epoch_relative", "Elect"]

# Remove duplicates for safety
keys_metric = table_metric_r[cols].drop_duplicates()
keys_fooof  = table_fooof_r[cols].drop_duplicates()


# Sort: Condition → Subject → Epoch
sort_cols = ["Condition_self", "Condition_emotion", "Condition_gaze", "Subject", "Epoch_relative"]

keys_metric = keys_metric.sort_values(sort_cols).reset_index(drop=True)
keys_fooof  = keys_fooof.sort_values(sort_cols).reset_index(drop=True)

# ---------- CHECK UNIQUE VALUES PER COLUMN ----------
print("🔎 Checking unique values per column:")
unique_ok = True

for col in cols:
    u_metric = set(keys_metric[col].unique())
    u_fooof  = set(keys_fooof[col].unique())

    if u_metric == u_fooof:
        print(f"  ✅ {col}: OK ({len(u_metric)} unique values)")
    else:
        unique_ok = False
        print(f"  ❌ {col}: DO NOT match")
        print(f"     Only in table_metric: {sorted(u_metric - u_fooof)}")
        print(f"     Only in table_fooof:  {sorted(u_fooof - u_metric)}")

# ---------- CHECK COMBINATIONS ----------
n_metric = len(keys_metric)
n_fooof = len(keys_fooof)

print(f"\nMetric rows: {n_metric}")
print(f"Fooof rows:  {n_fooof}")

if unique_ok and n_metric == n_fooof and keys_metric.merge(keys_fooof, on=cols).shape[0] == n_metric:
    print("✅ Combinations match exactly between both tables.")
else:
    print("❌ Combinations DO NOT match.")

    # Show combination differences
    only_metric = keys_metric.merge(
        keys_fooof, on=cols, how="left", indicator=True
    ).query("_merge == 'left_only'")

    only_fooof = keys_fooof.merge(
        keys_metric, on=cols, how="left", indicator=True
    ).query("_merge == 'left_only'")

    print(f"Rows only in table_metric_with_stimuli: {len(only_metric)}")
    print(f"Rows only in table_fooof:  {len(only_fooof)}")

🔎 Checking unique values per column:
  ✅ Subject: OK (30 unique values)
  ✅ event_id: OK (27 unique values)
  ✅ Condition_self: OK (3 unique values)
  ✅ Condition_emotion: OK (3 unique values)
  ✅ Condition_gaze: OK (3 unique values)
  ✅ Epoch_relative: OK (11 unique values)
  ✅ Elect: OK (59 unique values)

Metric rows: 369458
Fooof rows:  369458
✅ Combinations match exactly between both tables.


## Merge table_metric_to_merge with FOOOF table

In [13]:
#now, as the merge is going to be done with epoch relative, we remove Epoch from table fooof
table_fooof_r_clean = table_fooof_r.copy().drop(columns=["Epoch"])


# And now we merge
table_merged_df = pd.merge(
    table_metric_r, #table for metruc
    table_fooof_r_clean,
    on=["Subject", "Condition_self", "Condition_emotion", "Condition_gaze", "Epoch_relative", "Elect"],
    how="outer",
    indicator=True
)
## this code checks that there is no differences between both dataframes in the common columns

print(table_merged_df["_merge"].value_counts())
print(table_merged_df.columns)

_merge
both          369458
left_only          0
right_only         0
Name: count, dtype: int64
Index(['Subject', 'event_id_x', 'Condition_self', 'Condition_emotion',
       'Condition_gaze', 'Epoch', 'Elect', 'acw_50_elect_all_epoch_all',
       'acw_0_elect_all_epoch_all', 'Epoch_relative', 'event_id_y', 'delta',
       'theta', 'alpha', 'beta', 'gamma', 'offsets', 'exponents', 'r2',
       'error', '_merge'],
      dtype='object')


## Selection for final table

## Removal of unnecesary columns

In [14]:
# print(f"table_merged_df.columns are {table_merged_df.columns} ")

In [15]:
# table_merged_df

In [16]:
# # as the table is going to contain a HUGE AMOUNT of columns that are NOT NECESSARY for the analysis, Im not going to use them, Only for Layer_script event
# if layer_script=="event":
#        table_merged_df=table_merged_df.copy().drop(
#               columns=['Event_code','order_trial', 'Sample_tsv',
#               'Sample_difference', 'Sample_aligned', 'Sentence_tsv', 'Text_stimuli',
#               'number_item_original', 'match_ratio',
#               '_merge'])


#        print(f"After dropping columns,table_merged_df.columns are {table_merged_df.columns} ")

In [17]:
# # # # remove this, this is just to test things

# # table_metric_avg = (
# #     table_merged_df
# #     .groupby(["Subject", "Condition", "Epoch", "Epoch_relative"], as_index=False)
# #     .mean(numeric_only=True)
# # )

# # table_metric_avg_subj= table_metric_avg[table_metric_avg["Subject"]=="sub-V1001"].copy().drop(columns=['acw_50_elect_all_epoch_all',
# #        'acw_0_elect_all_epoch_all'])
# # table_metric_avg_subj


## Remove duplicated epochs
Only in event==layer

Another thing to take into account is that **epochs are duplicated**, specifically, **the epochs for questions appear as question**, but also without it. This means that the number "epoch" is not real, which is not entirely relevant, as we are NOT going to use epoch, but par_item, but its not a clear cut. 

So Im going to remove the epochs that are duplicated with and without questions, and then indexes of epoch and epoch relative  will be corrected. 

In [18]:
# if layer_script=="event":
#     df = table_merged_df.copy()
#     df["is_question"] = df["Condition"].astype(str).str.contains("question", case=False, na=False)

#     events = df[["Subject", "Condition", "Sample_events", "is_question"]].drop_duplicates()

#     events["Sample_events"] = pd.to_numeric(events["Sample_events"], errors="coerce")
#     events = events.dropna(subset=["Subject", "Sample_events"])
#     events["Sample_events"] = events["Sample_events"].astype("int64")

#     q_all  = events[events["is_question"]][["Subject", "Sample_events"]].copy()
#     nq_all = events[~events["is_question"]][["Subject", "Condition", "Sample_events"]].copy()

#     matched_list = []

#     for subj, nq in nq_all.groupby("Subject", sort=False):
#         q = q_all[q_all["Subject"] == subj][["Sample_events"]].copy()

#         if q.empty:
#             continue

#         nq = nq.sort_values("Sample_events").reset_index(drop=True)
#         q  = q.sort_values("Sample_events").reset_index(drop=True)

#         m = pd.merge_asof(
#             nq,
#             q.rename(columns={"Sample_events": "Sample_events_q"}),
#             left_on="Sample_events",
#             right_on="Sample_events_q",
#             tolerance=40,
#             direction="nearest"
#         )

#         # Ensure Subject exists (in case it gets dropped in some operations)
#         m["Subject"] = subj

#         matched_list.append(m)

#     matched = pd.concat(matched_list, ignore_index=True) if matched_list else pd.DataFrame(
#         columns=["Subject", "Condition", "Sample_events", "Sample_events_q"]
#     )

#     to_drop_events = matched.loc[matched["Sample_events_q"].notna(), ["Subject", "Condition", "Sample_events"]]

#     print("Non-question events to drop:", len(to_drop_events))

#     table_merged_cleaned_df = (
#         df.merge(to_drop_events.assign(_drop=1), on=["Subject", "Condition", "Sample_events"], how="left")
#         .query("_drop != 1")
#         .drop(columns=["_drop", "is_question"])
#     )

#     print("Rows before:", len(df))
#     print("Rows after:", len(table_merged_cleaned_df))

#     table_merged_cleaned_df


#     # ------------------------------------------------------------
#     # Rebuild Epoch per Subject based on Sample_events (min sample = Epoch 0)
#     # ------------------------------------------------------------

#     # Create a per-event mapping (Subject + Sample_events) -> new Epoch index
#     epoch_map = (
#         table_merged_cleaned_df[["Subject", "Sample_events"]]
#         .drop_duplicates()
#         .sort_values(["Subject", "Sample_events"])
#     )

#     epoch_map["Epoch"] = epoch_map.groupby("Subject").cumcount()

#     # Merge back so the new Epoch is replicated across all Elect rows
#     table_merged_cleaned_df = table_merged_cleaned_df.drop(columns=["Epoch"], errors="ignore").merge(
#         epoch_map,
#         on=["Subject", "Sample_events"],
#         how="left",
#         validate="m:1"
#     )


#     #now we recalculate the indexes in Epoch_relative, to avoid possible "jumps" as we eliminated epochs here
#     table_merged_cleaned_df=renumber_epochs(table_merged_cleaned_df)

    
# if layer_script=="block":
#     table_merged_cleaned_df = table_merged_df.copy()
    
    
# table_merged_cleaned_df.head()

In [19]:
## DELETE THIS

# # if layer_script=="block":
# #     table_clean_avg = (
# #         table_merged_cleaned_df
# #         .groupby(["Subject", "Condition", "Epoch", "Epoch_relative"], as_index=False)
# #         .mean(numeric_only=True)
# #     )

# #     table_clean_avg_subj= table_clean_avg[table_clean_avg["Subject"]=="sub-V1001"].copy().drop(columns=['acw_50_elect_all_epoch_all',
# #         'acw_0_elect_all_epoch_all'])


# #     table_clean_avg_subj

### Add specific columns (only for event)

For the LMM that we will apply with R, we need to put all the columns in a certain format, in order to have all the factors, wich will be
- **Condition**: zinnen or sentence (common in both block and event layer)

From event, we rename condition to **condition_specific**, wich will contain the full name (e.g. zinnen_RC_plus_question_hit) and we decompose it in the different parts
- **Condition**: that has zinnen or wordlist
- **Condition_RC**: which contains if there is relative clause (RC) or not
- **Condition_question**: which contains if there is question, and the answer

In [20]:
# #only changes in event
# if layer_script == "event":

#     # 1) Renombrar Condition -> Condition_specific
#     table_merged_cleaned_df = table_merged_cleaned_df.rename(columns={"Condition": "Condition_specific"})

#     # Asegurar Series (NO uses doble corchete aquí)
#     s = table_merged_cleaned_df["Condition_specific"].astype("string")

#     # 2) Nueva columna Condition (Woorden / Zinnen / <NA>)
#     table_merged_cleaned_df["Condition"] = pd.NA
#     mask_woorden = s.str.contains("woorden", case=False, na=False)
#     mask_zinnen  = s.str.contains("zinnen",  case=False, na=False)

#     table_merged_cleaned_df.loc[mask_woorden, "Condition"] = "woorden"
#     table_merged_cleaned_df.loc[mask_zinnen,  "Condition"] = "zinnen"

    
#     def to_Condition_RC(val):
#         if pd.isna(val):
#             return pd.NA

#         s = str(val).lower()

#         # base: woorden / zinnen
#         if "woorden" in s:
#             base = "woorden"
#         elif "zinnen" in s:
#             base = "zinnen"
#         else:
#             return pd.NA

#         # tipo RC
#         if "rc_plus" in s:
#             rc = "RC_plus"
#         elif "rc_neg" in s:
#             rc = "RC_neg"
#         else:
#             return pd.NA

#         return rc


#     table_merged_cleaned_df["Condition_RC"] = (
#         table_merged_cleaned_df["Condition_specific"]
#         .apply(to_Condition_RC)
#     )
    
    
#     def to_condition_question(val):
#         if pd.isna(val):
#             return pd.NA

#         s = str(val).lower()

#         # base
#         if "woorden" in s:
#             base = "woorden"
#         elif "zinnen" in s:
#             base = "zinnen"
#         else:
#             return pd.NA

#         # tipo de pregunta
#         if "question_hit" in s:
#             question = "hit"
#         elif "question_incorrect" in s:
#             question = "incorrect"
#         else:
#             return pd.NA

#         return question

#     table_merged_cleaned_df["Condition_question"] = (
#         table_merged_cleaned_df["Condition_specific"]
#         .apply(to_condition_question)
#     )

#         # 4) Reordenar columnas (dejando el resto al final)
#     desired_order = [
#         "Subject",
#         
#         "Condition",
#         
#         
    # ]

    # remaining_cols = [c for c in table_merged_cleaned_df.columns if c not in desired_order]

    # table_merged_cleaned_df = table_merged_cleaned_df[desired_order + remaining_cols]


# Its not necessary to do anything in block condition

In [21]:

# # table_clean_avg = (
# #     table_merged_cleaned_df
# #     .groupby(["Subject", "Condition", "Condition_question" ,"Epoch", "Epoch_relative"], as_index=False, dropna=False)
# #     .mean(numeric_only=True)
# # )

# # table_clean_avg_subj= table_clean_avg[table_clean_avg["Subject"]=="sub-V1001"].copy().drop(columns=['acw_50_elect_all_epoch_all',
# #     'acw_0_elect_all_epoch_all'])


# # table_clean_avg_subj

# Now we add the type of channel

## Not necessary in self

In [22]:
# dict_isc= pd.read_pickle(ISC_block_path /f"ISC_results_block.pkl")
# dict_woorden_block=dict_isc['dict_isc_WOORDEN']
# dict_zinnen_block = dict_isc['dict_isc_ZINNEN']
# ##channels significant adjusted in each condition
# numbers_channels_woorden= dict_woorden_block["significant_channels_adjusted"]
# numbers_channels_zinnen= dict_zinnen_block["significant_channels_adjusted"]
# print("numbers_channels_woorden", numbers_channels_woorden, "numbers_channels_zinnen", numbers_channels_zinnen)
# print("len(numbers_channels_woorden)",len(numbers_channels_woorden),  "len(numbers_channels_zinnen)",len(numbers_channels_zinnen))

# names_channels_woorden=dict_woorden_block["significant_channels_adjusted_names"]
# names_channels_zinnen=dict_zinnen_block["significant_channels_adjusted_names"]

# print("names_channels_woorden", names_channels_woorden)
# print("names_channels_zinnen", names_channels_zinnen)

# h_subj =0
# path_epochs = epochs_clean_path / f"{subjects[h_subj]}_epochs_{layer_script}-epo.fif"
# epochs = mne.read_epochs(path_epochs, preload=False)
# #establecimiento de canales palabras, canales frases y canales mixtos
# # Tomamos el orden original de los canales del objeto epochs
# all_channels = epochs.ch_names
# all_channels_numbers= [epochs.ch_names.index(ch) for ch in epochs.ch_names]


# del epochs

# # Palabras
# numbers_channels_only_woorden = [
#     ch for ch in all_channels_numbers if ch in numbers_channels_woorden and ch not in numbers_channels_zinnen
# ]

# numbers_channels_only_zinnen = [
#     ch for ch in all_channels_numbers if ch in numbers_channels_zinnen and ch not in numbers_channels_woorden
# ]

# numbers_channels_intersection = [
#     ch for ch in all_channels_numbers if ch in numbers_channels_woorden and ch in numbers_channels_zinnen
# ]

# # Lo mismo pero usando nombres
# names_channels_only_woorden = [
#     ch for ch in all_channels if ch in names_channels_woorden and ch not in names_channels_zinnen
# ]

# names_channels_only_zinnen = [
#     ch for ch in all_channels if ch in names_channels_zinnen and ch not in names_channels_woorden
# ]

# names_channels_intersection = [
#     ch for ch in all_channels if ch in names_channels_woorden and ch in names_channels_zinnen
# ]
# # ---------------------------
# # Print resumen
# # ---------------------------

# print(f"len(significant_channels_only_woorden): {len(numbers_channels_only_woorden)}, "
#       f"len(significant_channels_only_zinnen): {len(numbers_channels_only_zinnen)}, "
#       f"len(significant_channels_intersection): {len(numbers_channels_intersection)}")



# print("\n✅ Only woorden (names):", names_channels_only_woorden)
# print("✅ Only zinnen (names):", names_channels_only_zinnen)
# print("✅ Intersection (names):", names_channels_intersection)


# dict_select_channels={
#     "names_channels_only_woorden": names_channels_only_woorden,
#     "names_channels_only_zinnen": names_channels_only_zinnen,
#     "names_channels_intersection": names_channels_intersection
# }

In [23]:
# # Check channel membership sets
# set_woorden = set(names_channels_woorden)
# set_zinnen = set(names_channels_zinnen)

# def channel_labels(ch):
#     """
#     Returns:
#       - channel_type: intersection / only_woorden / only_zinnen / NaN
#       - channel_condition: woorden / zinnen / NaN
#     """
#     in_woorden = ch in set_woorden
#     in_zinnen  = ch in set_zinnen

#     # Channel_type (set relationship)
#     if in_woorden and in_zinnen:
#         channel_type = "channel_intersection"
#     elif in_woorden:
#         channel_type = "channel_only_woorden"
#     elif in_zinnen:
#         channel_type = "channel_only_zinnen"
#     else:
#         channel_type = np.nan

#     # Channel_condition (base membership)
#     if in_woorden:
#         channel_condition = "channel_woorden"
#     elif in_zinnen:
#         channel_condition = "channel_zinnen"
#     else:
#         channel_condition = np.nan

#     return channel_type, channel_condition


# # Create both columns using the same function
# tmp = table_merged_cleaned_df["Elect"].apply(channel_labels)

# table_merged_cleaned_df["Channel_type"] = tmp.apply(lambda x: x[0])
# table_merged_cleaned_df["Channel_condition"] = tmp.apply(lambda x: x[1])

# table_merged_cleaned_df.head()

In [24]:
# #Now we want to check if the channels were added correctly

# def assert_channel_sets(df, df_label, original_list, original_label):
#     """
#     This function checks whether the set of channels in the dataframe
#     (filtered by a specific channel condition) matches exactly the set
#     of channels provided in the original dictionary.

#     Parameters
#     ----------
#     df : DataFrame
#         The cleaned dataframe containing channel information.
#     df_label : str
#         The value used to filter the column 'Channel_type'.
#     original_list : list
#         The list of expected channel names from the original dictionary.
#     original_label : str
#         The label/name of the original dictionary (for reporting purposes).
#     """

#     # Extract the set of channels from the dataframe for the given condition
#     df_set = set(
#         df.query("Channel_type == @df_label")["Elect"]
#         .dropna()
#         .unique()
#     )

#     # Convert the expected channel list into a set
#     orig_set = set(original_list)

#     # Differences between dataframe channels and original dictionary channels
#     only_df = df_set - orig_set
#     only_orig = orig_set - df_set

#     print(f"\n🔎 CHECK {df_label}")
#     print(f"DataFrame count: {len(df_set)} | Original dictionary count: {len(orig_set)}")
#     print("Only in dataframe:", sorted(only_df))
#     print("Only in original dictionary:", sorted(only_orig))

#     # Assert ensures the script stops if there is any mismatch
#     assert not only_df and not only_orig, (
#         f"\n❌ Mismatch in {df_label}\n"
#         f"Only in dataframe: {sorted(only_df)}\n"
#         f"Only in original dictionary: {sorted(only_orig)}"
#     )

#     print(f"✅ {df_label} matches {original_label}")


# # ---------------------------------------------------
# # Run channel consistency checks
# # The script will stop automatically if a mismatch is detected
# # ---------------------------------------------------

# assert_channel_sets(
#     table_merged_cleaned_df,
#     "channel_only_woorden",
#     names_channels_only_woorden,
#     "names_channels_only_woorden"
# )

# assert_channel_sets(
#     table_merged_cleaned_df,
#     "channel_only_zinnen",
#     names_channels_only_zinnen,
#     "names_channels_only_zinnen"
# )

# assert_channel_sets(
#     table_merged_cleaned_df,
#     "channel_intersection",
#     names_channels_intersection,
#     "names_channels_intersection"
# )

# Change names from zinnen to sentence, from woorden to wordlist 
Now as we have the definite tables, we change the names 
We will also re order the columns


In [25]:
table_merged_cleaned_df=table_merged_df.copy()

In [26]:
table_merged_cleaned_df.columns

Index(['Subject', 'event_id_x', 'Condition_self', 'Condition_emotion',
       'Condition_gaze', 'Epoch', 'Elect', 'acw_50_elect_all_epoch_all',
       'acw_0_elect_all_epoch_all', 'Epoch_relative', 'event_id_y', 'delta',
       'theta', 'alpha', 'beta', 'gamma', 'offsets', 'exponents', 'r2',
       'error', '_merge'],
      dtype='object')

In [27]:


table_merged_cleaned_df = table_merged_cleaned_df.rename(
    columns={"Elect": "Channel"}
)

if layer_script=="event":
    # Desired column order

    if dynamic==False:

        
        desired_order = [
            "Subject",
            "Condition_self", 
            "Condition_emotion", 
            "Condition_gaze",
            "Epoch",
            "Epoch_relative",
            # "onset",
            # "Sample_events",
            # "number_item",
            # "par_item",
            "Channel",

            "acw_50_elect_all_epoch_all",
            "acw_0_elect_all_epoch_all",
            "delta",
            "theta",
            "alpha",
            "beta",
            "gamma",
            "offsets",
            "exponents",
            "r2",
            "error",
        ]

        # Keep any remaining columns at the end (safe version)
        remaining_cols = [c for c in table_merged_cleaned_df.columns if c not in desired_order]

        # Reorder dataframe
        table_merged_cleaned_df = table_merged_cleaned_df[desired_order + remaining_cols]
        
    elif dynamic==True:
        desired_order = [
        "Subject",
        
        "Condition_self", 
        "Condition_emotion", 
        "Condition_gaze",
        "Epoch",
        "Epoch_relative",
        # "onset",
        # "Sample_events",
        # "number_item",
        # "par_item",
        "Channel",
        "Channel_condition",
        "Channel_type",

        # ACW
        "acw_50_slope_elect_all_epoch_all",
        "acw_50_std_elect_all_epoch_all",
        "acw_0_slope_elect_all_epoch_all",
        "acw_0_std_elect_all_epoch_all",

        # Band powers (slope + std, one by one)
        "delta_slope",
        "delta_std",
        "theta_slope",
        "theta_std",
        "alpha_slope",
        "alpha_std",
        "beta_slope",
        "beta_std",
        "gamma_slope",
        "gamma_std",

        # FOOOF params (slope + std, one by one)
        "offsets_slope",
        "offsets_std",
        "exponents_slope",
        "exponents_std"
        ]

        
        
# elif layer_script == "block":

#     desired_order = [
#         "Subject",
#         "Condition_self", "Condition_emotion", "Condition_gaze",
#         "Epoch",
#         "Epoch_relative",
#         "Channel",
#         "Channel_condition",
#         "Channel_type",
#         "acw_50_elect_all_epoch_all",
#         "acw_0_elect_all_epoch_all",
#         "delta",
#         "theta",
#         "alpha",
#         "beta",
#         "gamma",
#         "offsets",
#         "exponents",
#         "r2",
#         "error",
#         "_merge",
#     ]

# keep remaining columns at the end (safe version)
remaining_cols = [c for c in table_merged_cleaned_df.columns if c not in desired_order]

table_merged_cleaned_df = table_merged_cleaned_df[desired_order + remaining_cols]
    

In [28]:
table_merged_cleaned_df

,Subject,Condition_self,Condition_emotion,Condition_gaze,Epoch,Epoch_relative,Channel,acw_50_elect_all_epoch_all,acw_0_elect_all_epoch_all,delta,...,alpha,beta,gamma,offsets,exponents,r2,error,event_id_x,event_id_y,_merge
0,s01b,friend,negative,1,152,0,AF3,0.031250,0.152344,0.0,...,0.000000,0.882773,0.000000,-11.407993,1.080874,0.864982,0.190538,46,46,both
1,s01b,friend,negative,1,152,0,AF4,0.031250,0.171875,0.0,...,0.774850,0.606128,0.000000,-10.791569,1.497124,0.907722,0.164664,46,46,both
2,s01b,friend,negative,1,152,0,AF7,0.031250,0.140625,0.0,...,0.000000,0.541175,0.554864,-11.229305,1.547114,0.914000,0.150064,46,46,both
3,s01b,friend,negative,1,152,0,AF8,0.015625,0.226562,0.0,...,0.000000,0.861546,0.802711,-11.340884,1.060545,0.864093,0.131341,46,46,both
4,s01b,friend,negative,1,152,0,C1,0.031250,0.191406,0.0,...,1.058257,1.027224,0.000000,-10.685160,2.091084,0.917919,0.205921,46,46,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
369453,s36b,unknown,positive,3,735,4,Pz,0.011719,0.156250,0.0,...,0.000000,0.853515,0.992001,-10.964776,1.232385,0.763978,0.199484,94,94,both
369454,s36b,unknown,positive,3,735,4,T7,0.007812,0.015625,0.0,...,0.000000,0.890339,0.000000,-11.773773,0.425150,0.231929,0.281973,94,94,both
369455,s36b,unknown,positive,3,735,4,T8,0.007812,0.011719,0.0,...,0.000000,0.865770,0.617869,-12.708092,-0.704268,0.462547,0.284129,94,94,both
369456,s36b,unknown,positive,3,735,4,TP7,0.007812,0.011719,0.0,...,0.000000,1.065875,0.732294,-12.034926,0.054176,0.474968,0.220603,94,94,both


In [29]:
suffix_fooof_str

'self_filt_1-40_fixed_event'

In [30]:
# -------------------------------
# Save merged table
# -------------------------------
if dynamic:
    output_path = ACW_path / f"table_merged_dynamic_ACW_FOOOF_results_{suffix_fooof_str}.csv"
else:
    output_path = ACW_path / f"table_merged_ACW_FOOOF_{suffix_fooof_str}.csv"

table_merged_cleaned_df.to_csv(output_path, index=False)

print(f"Saved to {output_path}")

Saved to g:\PROYECTO_SELF\SELF_visual\output_analysis\analysis_event\acw_event\table_merged_ACW_FOOOF_self_filt_1-40_fixed_event.csv
